<a href="https://colab.research.google.com/github/Shibu4064/ESSEX-THESIS/blob/main/1vsall_Statistics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix

# ============================================
# Load Training Data
# ============================================
print("Loading training data...")
train_df = pd.read_excel('/content/Human labelled_DTU.xlsx', skiprows=1)
train_df.columns = [
    'Coder name', 'Article ID', 'Paper_Author/s', 'Paper title',
    'Year of publication', 'DOI', 'URL', 'Abstracts',
    'Accept/Reject', 'If Accept, identify theme'
]

train_df = train_df[train_df['Accept/Reject'].isin(['Accept', 'Reject'])].copy()
train_df['text'] = train_df['Abstracts'].fillna('')
train_df = train_df[train_df['text'].str.len() > 50].reset_index(drop=True)

# Ground truth
train_true_labels = (train_df['Accept/Reject'] == 'Accept').astype(int)

# ============================================
# Load Test Predictions
# ============================================
test_predictions_df = pd.read_csv('/content/reverse_strategy_predictions.csv')

# ============================================
# COMPARISON ANALYSIS
# ============================================
print("\n" + "="*80)
print("COMPARISON: TRAINING vs TEST PREDICTIONS")
print("="*80)

print("\n1. DISTRIBUTION COMPARISON")
print("-"*80)

# Training statistics
train_n_accept = (train_true_labels == 1).sum()
train_n_reject = (train_true_labels == 0).sum()
train_accept_rate = train_n_accept / len(train_df) * 100

# Test statistics
test_n_accept = (test_predictions_df['Prediction'] == 'Accept').sum()
test_n_reject = (test_predictions_df['Prediction'] == 'Reject').sum()
test_accept_rate = test_n_accept / len(test_predictions_df) * 100

print(f"\n{'Metric':<30} | Training (True) | Test (Predicted)")
print(f"{'-'*30}|-----------------|------------------")
print(f"{'Total papers':<30} | {len(train_df):15,} | {len(test_predictions_df):16,}")
print(f"{'Accept papers':<30} | {train_n_accept:15,} | {test_n_accept:16,}")
print(f"{'Reject papers':<30} | {train_n_reject:15,} | {test_n_reject:16,}")
print(f"{'Accept rate':<30} | {train_accept_rate:14.2f}% | {test_accept_rate:15.2f}%")
print(f"{'Accept:Reject ratio':<30} | 1:{train_n_reject/train_n_accept:13.1f} | 1:{test_n_reject/test_n_accept:14.1f}")

# ============================================
# 2. THEME DISTRIBUTION COMPARISON
# ============================================
print("\n2. THEME DISTRIBUTION COMPARISON")
print("-"*80)

# Training theme distribution
train_accepted = train_df[train_df['Accept/Reject'] == 'Accept'].copy()
train_accepted = train_accepted[train_accepted['If Accept, identify theme'].notna()]
train_theme_counts = train_accepted['If Accept, identify theme'].value_counts()

# Test theme distribution
test_accepted = test_predictions_df[test_predictions_df['Prediction'] == 'Accept']
test_theme_counts = test_accepted['Theme'].value_counts()

print(f"\n{'Theme':<55} | Train % | Test %")
print(f"{'-'*55}|---------|--------")

# Get all unique themes
all_themes = set(train_theme_counts.index) | set(test_theme_counts.index)

for theme in sorted(all_themes):
    train_pct = (train_theme_counts.get(theme, 0) / len(train_accepted) * 100) if len(train_accepted) > 0 else 0
    test_pct = (test_theme_counts.get(theme, 0) / len(test_accepted) * 100) if len(test_accepted) > 0 else 0

    diff = abs(train_pct - test_pct)
    marker = "✓" if diff < 5 else "⚠" if diff < 10 else "❌"

    print(f"{theme[:54]:<55} | {train_pct:6.1f}% | {test_pct:6.1f}% {marker}")

print(f"\n✓ = Good match (< 5% diff)")
print(f"⚠ = Slight deviation (5-10% diff)")
print(f"❌ = Large deviation (> 10% diff)")

# ============================================
# 3. ASSESSMENT
# ============================================
print("\n3. OVERALL ASSESSMENT")
print("-"*80)

# Check accept rate similarity
rate_diff = abs(train_accept_rate - test_accept_rate)
if rate_diff < 3:
    print(f"\n✓ Accept Rate: EXCELLENT match ({rate_diff:.1f}% difference)")
elif rate_diff < 5:
    print(f"\n✓ Accept Rate: Good match ({rate_diff:.1f}% difference)")
elif rate_diff < 10:
    print(f"\n⚠️ Accept Rate: Moderate deviation ({rate_diff:.1f}% difference)")
else:
    print(f"\n❌ Accept Rate: Large deviation ({rate_diff:.1f}% difference)")
    if test_accept_rate < train_accept_rate:
        print(f"   → Model is too conservative. Too many papers rejected.")
    else:
        print(f"   → Model is too permissive. Too many papers accepted.")

# Check theme diversity
n_themes_train = len(train_theme_counts)
n_themes_test = len(test_theme_counts)

print(f"\nTheme Diversity:")
print(f"  Training: {n_themes_train}/12 themes")
print(f"  Test:     {n_themes_test}/12 themes")

if n_themes_test >= n_themes_train - 1:
    print(f"  ✓ Excellent! Similar diversity to training.")
elif n_themes_test >= 8:
    print(f"  ✓ Good diversity maintained.")
elif n_themes_test >= 5:
    print(f"  ⚠️ Moderate diversity. Some themes missing.")
else:
    print(f"  ❌ Poor diversity. Model collapsed!")

print("\n" + "="*80)

Loading training data...

COMPARISON: TRAINING vs TEST PREDICTIONS

1. DISTRIBUTION COMPARISON
--------------------------------------------------------------------------------

Metric                         | Training (True) | Test (Predicted)
------------------------------|-----------------|------------------
Total papers                   |           1,719 |           10,175
Accept papers                  |             199 |            4,806
Reject papers                  |           1,520 |            5,369
Accept rate                    |          11.58% |           47.23%
Accept:Reject ratio            | 1:          7.6 | 1:           1.1

2. THEME DISTRIBUTION COMPARISON
--------------------------------------------------------------------------------

Theme                                                   | Train % | Test %
-------------------------------------------------------|---------|--------
Access to Services and Wellbeing                        |    3.0% |    3.5% ✓
Bet

/usr/local/lib/python3.12/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


In [2]:
import pandas as pd
import numpy as np

# Load your predictions
predictions_df = pd.read_csv('/content/reverse_strategy_predictions.csv')

print("="*80)
print("BINARY CLASSIFICATION ANALYSIS (FROM PREDICTIONS CSV)")
print("="*80)

# ============================================
# 1. BASIC STATISTICS
# ============================================
print("\n1. PREDICTION DISTRIBUTION")
print("-"*80)

n_total = len(predictions_df)
n_accept = (predictions_df['Prediction'] == 'Accept').sum()
n_reject = (predictions_df['Prediction'] == 'Reject').sum()

print(f"\nTotal papers: {n_total:,}")
print(f"Predicted Accept: {n_accept:,} ({n_accept/n_total*100:.2f}%)")
print(f"Predicted Reject: {n_reject:,} ({n_reject/n_total*100:.2f}%)")
print(f"Accept/Reject Ratio: 1:{n_reject/max(n_accept,1):.1f}")

# ============================================
# 2. CONFIDENCE ANALYSIS
# ============================================
print("\n2. CONFIDENCE STATISTICS")
print("-"*80)

accept_rows = predictions_df[predictions_df['Prediction'] == 'Accept']
reject_rows = predictions_df[predictions_df['Prediction'] == 'Reject']

print(f"\nAccept Predictions (n={len(accept_rows):,}):")
print(f"  Mean confidence:   {accept_rows['Confidence'].mean():.4f}")
print(f"  Median confidence: {accept_rows['Confidence'].median():.4f}")
print(f"  Std confidence:    {accept_rows['Confidence'].std():.4f}")
print(f"  Min confidence:    {accept_rows['Confidence'].min():.4f}")
print(f"  Max confidence:    {accept_rows['Confidence'].max():.4f}")

# Confidence bins for Accept
print(f"\n  Confidence Distribution (Accept):")
bins = [0.0, 0.3, 0.5, 0.7, 0.9, 1.0]
for i in range(len(bins)-1):
    count = ((accept_rows['Confidence'] >= bins[i]) & (accept_rows['Confidence'] < bins[i+1])).sum()
    print(f"    {bins[i]:.1f}-{bins[i+1]:.1f}: {count:5d} papers ({count/len(accept_rows)*100:5.1f}%)")

print(f"\nReject Predictions (n={len(reject_rows):,}):")
print(f"  Mean confidence:   {reject_rows['Confidence'].mean():.4f}")
print(f"  Median confidence: {reject_rows['Confidence'].median():.4f}")
print(f"  Std confidence:    {reject_rows['Confidence'].std():.4f}")
print(f"  Min confidence:    {reject_rows['Confidence'].min():.4f}")
print(f"  Max confidence:    {reject_rows['Confidence'].max():.4f}")

# ============================================
# 3. THEME DISTRIBUTION (for Accept only)
# ============================================
print("\n3. THEME DISTRIBUTION (Accepted Papers)")
print("-"*80)

if len(accept_rows) > 0:
    theme_counts = accept_rows['Theme'].value_counts()
    n_themes_predicted = len(theme_counts)

    print(f"\nNumber of different themes predicted: {n_themes_predicted}/12")
    print(f"\nTheme breakdown:")
    for i, (theme, count) in enumerate(theme_counts.items(), 1):
        pct = count / len(accept_rows) * 100
        print(f"  {i:2d}. {count:4d} papers ({pct:5.1f}%) - {theme}")

    # Check diversity
    if n_themes_predicted >= 10:
        print(f"\n✓ Excellent diversity! {n_themes_predicted}/12 themes predicted.")
    elif n_themes_predicted >= 7:
        print(f"\n✓ Good diversity! {n_themes_predicted}/12 themes predicted.")
    elif n_themes_predicted >= 5:
        print(f"\n⚠️ Moderate diversity. {n_themes_predicted}/12 themes predicted.")
    else:
        print(f"\n❌ Poor diversity! Only {n_themes_predicted}/12 themes predicted.")
        print(f"   Model may have collapsed.")

# ============================================
# 4. SANITY CHECKS
# ============================================
print("\n4. SANITY CHECKS")
print("-"*80)

# Expected accept rate from training: ~11-12%
expected_accept_rate = 11.5
actual_accept_rate = n_accept / n_total * 100

print(f"\nAccept Rate:")
print(f"  Expected (from training): ~{expected_accept_rate:.1f}%")
print(f"  Actual (test predictions): {actual_accept_rate:.2f}%")

diff = abs(actual_accept_rate - expected_accept_rate)
if diff < 3:
    print(f"  ✓ Good! Within expected range.")
elif diff < 7:
    print(f"  ⚠️ Slight deviation but acceptable.")
else:
    print(f"  ❌ Large deviation! Model might be miscalibrated.")

# Check for empty themes
empty_themes = (accept_rows['Theme'] == '').sum()
if empty_themes > 0:
    print(f"\n⚠️ Warning: {empty_themes} accepted papers have no theme assigned!")

# Check confidence makes sense
if accept_rows['Confidence'].mean() < 0.4:
    print(f"\n⚠️ Warning: Low average confidence ({accept_rows['Confidence'].mean():.3f}) for Accept predictions.")
    print(f"   Model may be uncertain. Consider adjusting threshold.")

print("\n" + "="*80)

BINARY CLASSIFICATION ANALYSIS (FROM PREDICTIONS CSV)

1. PREDICTION DISTRIBUTION
--------------------------------------------------------------------------------

Total papers: 10,175
Predicted Accept: 4,806 (47.23%)
Predicted Reject: 5,369 (52.77%)
Accept/Reject Ratio: 1:1.1

2. CONFIDENCE STATISTICS
--------------------------------------------------------------------------------

Accept Predictions (n=4,806):
  Mean confidence:   0.8553
  Median confidence: 0.8098
  Std confidence:    0.1004
  Min confidence:    0.5168
  Max confidence:    0.9995

  Confidence Distribution (Accept):
    0.0-0.3:     0 papers (  0.0%)
    0.3-0.5:     0 papers (  0.0%)
    0.5-0.7:   267 papers (  5.6%)
    0.7-0.9:  2648 papers ( 55.1%)
    0.9-1.0:  1891 papers ( 39.3%)

Reject Predictions (n=5,369):
  Mean confidence:   0.4999
  Median confidence: 0.5138
  Std confidence:    0.1086
  Min confidence:    0.2187
  Max confidence:    0.7949

3. THEME DISTRIBUTION (Accepted Papers)
--------------------